## Encode Data Kategorikal

In [12]:
import pandas as pd

try:
    df = pd.read_csv('../Dataset/dataset_harga_buku.csv')
    print(df.head())
except FileNotFoundError:
    print("Error: File tidak ditemukan.")
    exit()

   jumlah_halaman jenis_cover kualitas_cetak     penerbit   harga
0             202   Softcover      Bookpaper     Erlangga   82500
1             448   Softcover      Bookpaper   Buku Mojok  109000
2             370   Softcover      Bookpaper  Marjin Kiri  104000
3             206   Softcover      Bookpaper     Erlangga   87000
4             171   Softcover      Bookpaper   Buku Mojok   75000


In [13]:
# Kolom yang di-encode: 'jenis_cover', 'kualitas_cetak', dan 'penerbit'
# drop_first=True akan menghapus 1 kategori dari tiap fitur untuk dijadikan baseline (nilai 0)
fitur_kategorikal = ['jenis_cover', 'kualitas_cetak', 'penerbit']

df_encoded = pd.get_dummies(
    df, 
    columns=fitur_kategorikal, 
    drop_first=True, 
    dtype=int  # Mengubah hasil boolean (True/False) langsung menjadi biner (1/0)
)

# Menata ulang posisi kolom agar target 'harga' berada di paling kanan
kolom_fitur = [col for col in df_encoded.columns if col != 'harga']
df_encoded = df_encoded[kolom_fitur + ['harga']]

# Hasil setelah encoding
print("\n--- Daftar Kolom Baru Hasil Encoding ---")
for i, col in enumerate(df_encoded.columns):
    print(f"{i+1}. {col}")

# Menyimpan hasil encoding ke file CSV baru untuk digunakan saat training model
df_encoded.to_csv('../Dataset/dataset_harga_buku_encoded.csv', index=False)
print("Data hasil encoding telah disimpan dengan nama: 'dataset_harga_buku_encoded.csv'")


--- Daftar Kolom Baru Hasil Encoding ---
1. jumlah_halaman
2. jenis_cover_Softcover
3. kualitas_cetak_Bookpaper
4. kualitas_cetak_HVS
5. penerbit_Erlangga
6. penerbit_Gramedia Pustaka Utama
7. penerbit_Marjin Kiri
8. penerbit_Mizan
9. harga
Data hasil encoding telah disimpan dengan nama: 'dataset_harga_buku_encoded.csv'


## Regresi Linear Berganda

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Membaca dataset hasil encoding
try:
    df = pd.read_csv('../Dataset/dataset_harga_buku_encoded.csv')
except FileNotFoundError:
    print("Error: File 'dataset_harga_buku_encoded.csv' tidak ditemukan.")
    print("Pastikan Anda sudah menjalankan script encoding terlebih dahulu.")
    exit()

# Memisahkan Fitur (X) dan Target/Label (y)
X = df.drop(columns=['harga'])
y = df['harga']

# Membagi data menjadi Data Training (80%) dan Data Testing (20%)
# random_state dikunci pada angka 42 agar hasil pembagian data selalu konsisten jika dijalankan ulang
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Jumlah data training : {X_train.shape[0]} baris")
print(f"Jumlah data testing  : {X_test.shape[0]} baris")

# Inisialisasi dan Pelatihan Model Regresi Linear Berganda
model = LinearRegression()
model.fit(X_train, y_train)
print("=== TRAINING SELESAI ===")

# Evaluasi Model menggunakan Data Testing
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n--- Hasil Evaluasi Performa Model ---")
print(f"R-squared (R2 Score)        : {r2:.4f} (Mendekati 1.0 berarti sangat akurat)")
print(f"Mean Absolute Error (MAE)   : Rp {mae:.2f} (Rata-rata pergeseran nilai prediksi dari harga asli)")

Jumlah data training : 160 baris
Jumlah data testing  : 40 baris
=== TRAINING SELESAI ===

--- Hasil Evaluasi Performa Model ---
R-squared (R2 Score)        : 0.9487 (Mendekati 1.0 berarti sangat akurat)
Mean Absolute Error (MAE)   : Rp 3382.61 (Rata-rata pergeseran nilai prediksi dari harga asli)


## Konversi C++

In [16]:
# Ekstraksi Bobot Matematis untuk Implementasi di ESP32
intercept = model.intercept_
coefficients = model.coef_
nama_fitur = X.columns.tolist()

with open("../patternHargaBuku/include/model_regresi.h", "w") as f:
    f.write("// File header hasil konversi model Regresi Linear Berganda\n")
    f.write("#ifndef MODEL_REGRESI_H\n")
    f.write("#define MODEL_REGRESI_H\n\n")
    
    # Menulis Intercept
    f.write(f"const float INTERCEPT = {intercept:.4f};\n\n")
    
    # Menulis Koefisien
    f.write("// Nilai Koefisien / Bobot per Fitur\n")
    for fitur, koefisien in zip(nama_fitur, coefficients):
        var_name = fitur.replace(" ", "_").replace("-", "_")
        f.write(f"const float COEF_{var_name} = {koefisien:.4f};\n")
        
    f.write("\n#endif // MODEL_REGRESI_H\n")

print("\nFile 'model_regresi.h' telah dibuat dan di simpan di folder 'include'.")


File 'model_regresi.h' telah dibuat dan di simpan di folder 'include'.
